In [1]:
import numpy as np
from numba import cuda
import math
import time

# ----------------------------------
# Generate Dataset (10 Million Values)
# ----------------------------------
N = 10_000_000

data = np.random.rand(N).astype(np.float32)

# Copy to GPU
d_data = cuda.to_device(data)

# ----------------------------------
# CUDA Kernel for Block-wise Reduction
# ----------------------------------
@cuda.jit
def max_reduce_kernel(input_arr, output_arr):

    # Shared memory for reduction
    shared = cuda.shared.array(256, dtype=np.float32)

    tid = cuda.threadIdx.x
    idx = cuda.blockIdx.x * cuda.blockDim.x + tid

    # Load data into shared memory
    if idx < input_arr.size:
        shared[tid] = input_arr[idx]
    else:
        shared[tid] = -1e20

    cuda.syncthreads()

    # Parallel Reduction
    stride = cuda.blockDim.x // 2

    while stride > 0:
        if tid < stride:
            shared[tid] = max(shared[tid], shared[tid + stride])

        cuda.syncthreads()
        stride //= 2

    # First thread stores block maximum
    if tid == 0:
        output_arr[cuda.blockIdx.x] = shared[0]

# ----------------------------------
# Multi-stage Reduction
# ----------------------------------
def gpu_max(arr):

    threads_per_block = 256

    current = cuda.to_device(arr)

    while current.size > 1:

        blocks = math.ceil(current.size / threads_per_block)

        output = cuda.device_array(blocks, dtype=np.float32)

        max_reduce_kernel[blocks, threads_per_block](current, output)

        current = output

    return current.copy_to_host()[0]

# ----------------------------------
# GPU Execution
# ----------------------------------
start_gpu = time.time()

gpu_result = gpu_max(data)

cuda.synchronize()

gpu_time = time.time() - start_gpu

# ----------------------------------
# CPU Execution
# ----------------------------------
start_cpu = time.time()

cpu_result = np.max(data)

cpu_time = time.time() - start_cpu

# ----------------------------------
# Results
# ----------------------------------
print(f"GPU Maximum : {gpu_result}")
print(f"CPU Maximum : {cpu_result}")

print(f"GPU Time : {gpu_time:.6f} sec")
print(f"CPU Time : {cpu_time:.6f} sec")

print(f"Difference : {abs(gpu_result - cpu_result)}")

GPU Maximum : 1.0
CPU Maximum : 1.0
GPU Time : 2.484396 sec
CPU Time : 0.009279 sec
Difference : 0.0


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
